**Data Cleaning**


import file hasil scraping

In [ ]:
import os, re
import pandas as pd
from google.colab import files

print("Pilih SEMUA file CSV hasil scraping")
unggah = files.upload()

berkas = [f for f in unggah if f.lower().endswith(".csv")]
if not berkas:
    raise SystemExit("Tidak ada CSV terbaca. Jalankan ulang sel ini dan pilih file-nya.")

print(f"\n{len(berkas)} berkas terbaca:")
for f in berkas:
    print(" -", f)

konfigurasi wilayah

In [ ]:
TAHUN_MIN = 2019                 # buang unggahan sebelum tahun ini
WILAYAH_SEKITAR = False          # True = ikut sertakan Bekasi/Depok/Tangerang
BUANG_LAPORAN_PERBAIKAN = True   # buang publikasi perbaikan oleh instansi
BUANG_TROTOAR = True
BUANG_JPO = True
BUANG_TOL = True

WILAYAH_KEYWORDS = {
    "Jakarta Pusat": [
        "jakarta pusat", "jakpus", "menteng", "gambir", "senen", "cempaka putih",
        "johar baru", "kemayoran", "sawah besar", "tanah abang", "medan merdeka",
        "gunung sahari", "thamrin", "monas", "pasar baru", "salemba", "kwitang",
        "kramat raya",
    ],
    "Jakarta Utara": [
        "jakarta utara", "jakut", "tanjung priok", "koja", "cilincing",
        "penjaringan", "pademangan", "kelapa gading", "sunter", "ancol",
        "marunda", "enggano", "yos sudarso", "boulevard raya", "gedong panjang",
        "plumpang",
    ],
    "Jakarta Barat": [
        "jakarta barat", "jakbar", "cengkareng", "kalideres", "grogol", "tambora",
        "taman sari", "kebon jeruk", "palmerah", "kembangan", "tomang", "slipi",
        "glodok", "s parman", "daan mogot", "kedoya", "meruya", "joglo",
        "semanan", "roxy",
    ],
    "Jakarta Selatan": [
        "jakarta selatan", "jaksel", "kebayoran", "tebet", "mampang", "pancoran",
        "jagakarsa", "pasar minggu", "cilandak", "pesanggrahan", "setiabudi",
        "kemang", "manggarai", "lenteng agung", "rawajati", "kuningan", "blok m",
        "fatmawati", "antasari", "gatot subroto", "gatsu", "simatupang",
        "radio dalam", "senopati", "casablanca", "warung buncit", "ragunan",
        "pondok indah", "rasuna said", "sudirman",
    ],
    "Jakarta Timur": [
        "jakarta timur", "jaktim", "matraman", "jatinegara", "cakung",
        "duren sawit", "kramat jati", "makasar", "pulo gadung", "pulogadung",
        "ciracas", "cipayung", "pasar rebo", "otista", "pramuka", "rawamangun",
        "klender", "cawang", "kebon nanas", "pondok bambu", "halim", "cibubur",
        "penggilingan", "buaran", "pulo gebang", "radjiman", "kalimalang",
    ],
    "Kep. Seribu": ["kepulauan seribu", "pulau seribu"],
}

KOTA_SEKITAR = ["bekasi", "tangerang", "depok", "bogor", "tangsel", "cikarang",
                "bintaro", "ciputat"]
KOTA_JAUH = ["bandung", "cirebon", "karawang", "cikampek", "sukabumi", "lampung",
             "surabaya", "semarang", "medan", "jawa tengah", "bitung", "balaraja",
             "antapani", "pantura", "japek", "bocimi", "merak", "sumatra",
             "kaltim", "papua", "aceh", "kalimantan", "jogja", "solo", "bali",
             "ntt", "flores"]

print("Konfigurasi dimuat.")

teks pattern & filtering function

In [ ]:
JALAN = r"(?:jalanan|jalan|jln\.?|jl\.?|aspal|ruas|badan jalan|beton|cor)"
RUSAK = (r"(?:rusak|berlubang|bolong|lubang|ambles|amblas|retak|bergelombang|"
         r"menganga|keriting|gronjal|hancur)")
OBJEK_LAIN = (
    r"(?:jpo|jembatan penyeberangan|jembatan penyebrangan|jembatan|cctv|kamera|"
    r"sekolah|gedung|warteg|kios|lapak|gardu|halte|kabel|pagar|atap|plafon|rumah|"
    r"ruko|bus|truk|mobil|motor|kereta|krl|kapal|saluran|got|trotoar|kanstin|"
    r"barrier|lampu|pju|tiang|billboard|baliho|spanduk|taman|pot|median|tanaman|"
    r"ac|mesin|ban|kendaraan|hp|laptop|kamar|kantor|pabrik|rem|kopling|aki)"
)

RE_LOKATIF = re.compile(
    r"(?:\bdi\b|\bpada\b|\bsepanjang\b|\bkawasan\b|\bdepan\b|\bseberang\b|"
    r"\bsekitar\b|\bdekat\b|\barah\b)\s*$", re.I)
RE_RUSAK = re.compile(rf"\b{RUSAK}\b", re.I)
RE_JALAN = re.compile(rf"\b{JALAN}\b", re.I)
RE_OBJEK = re.compile(rf"\b{OBJEK_LAIN}\b", re.I)
RE_MAJU  = re.compile(rf"^[ ,]*(?:di|pada|pd)?\s*(?:ruas\s+)?{JALAN}\b", re.I)
RE_IDIOM = re.compile(
    r"\b(?:siang bolong|tembok bolong|gali lubang|tutup lubang|lubang berjalan|"
    r"jalan kaki|jalan-jalan|jalan2|turun ke jalan|jalan hidup|jalan cerita|"
    r"jalan keluar|jalan tengah|jalan pintas|jalan buntu|jalan damai|"
    r"jalan panjang|jalan terus|jalan pikiran|setengah jalan|jalan sehat|"
    r"jalan santai|sepanjang jalan)\b", re.I)

RE_PERBAIKAN = re.compile(
    r"\b(?:diperbaiki|perbaikan|memperbaiki|perbaiki|ditambal|penambalan|menambal|"
    r"diaspal|pengaspalan|rampung|selesai dikerjakan|pasukan kuning|satgas|coldmix|"
    r"sudah diperbaiki|dikerjakan|pemeliharaan rutin|sudin bina marga|"
    r"suku dinas bina marga|sdbm|pengaturan dan pengamanan)\b", re.I)
RE_KELUHAN = re.compile(
    r"\b(?:tolong|mohon|keluhan|dikeluhkan|mengeluh|kapan|belum juga|tak kunjung|"
    r"tidak kunjung|dibenerin|segera|minta|rawan|bahaya|membahayakan|hati-?hati|"
    r"hati2|waspada|korban|tewas|meninggal|kecelakaan|please)\b", re.I)

STOPWORD_JALAN = (
    r"(?:depan|deket|dekat|arah|sebelah|menuju|sampai|tepatnya|tepat|yang|yg|udah|"
    r"sudah|tolong|mohon|banget|bgt|parah|rusak|bolong|berlubang|hancur|ambles|tuh|"
    r"nih|dong|dan|cc|via|kondisi|akibat|dihimbau|himbau|pengguna|tersebut|agar|"
    r"kembali|penghubung|tampak|mulai|akan|bisa|sering|kerap|banyak|hingga|serta|"
    r"selesai|telah|oleh|untuk|pada|dari|masih|juga|hati)\b")
KATA_BUKAN_JALAN = {"raya", "ini", "itu", "di", "ke", "jakarta"}


def deteksi_wilayah(teks):
    """Tebak kota administrasi DKI dari nama kawasan yang disebut di teks."""
    teks = f" {str(teks).lower()} "
    cocok = [w for w, kws in WILAYAH_KEYWORDS.items() if any(k in teks for k in kws)]
    if len(cocok) == 1:
        return cocok[0]
    if len(cocok) > 1:
        return "AMBIGU: " + " | ".join(cocok)
    return ""


def cari_kota(teks, daftar):
    teks = str(teks).lower()
    return ", ".join(sorted({k for k in daftar if k in teks}))


def ekstrak_nama_jalan(teks):
    """Ambil nama jalan setelah penanda 'Jl./Jalan'."""
    teks = re.sub(r"https?://\S+", "", str(teks))
    hasil = []
    pola = r"(?:jl\.?|jln\.?|jalan)\s+([A-Za-z0-9\.\-\s]{3,45})"
    for kandidat in re.findall(pola, teks, flags=re.I):
        kandidat = re.split(STOPWORD_JALAN, kandidat, flags=re.I)[0]
        kandidat = re.sub(r"\s+", " ", kandidat).strip(" .-,")
        if len(kandidat) >= 3 and kandidat.lower() not in KATA_BUKAN_JALAN:
            hasil.append(kandidat.title())
    return " ; ".join(dict.fromkeys(hasil))


def subjek_adalah_jalan(teks):
    """
    True bila kata kerusakan dalam teks memang merujuk pada jalan.
      1. Kata benda terdekat sebelum kata kerusakan jadi kandidat subjek.
         "CCTV di Jalan Pejompongan rusak" -> subjeknya CCTV, bukan jalan.
      2. Frasa lokatif ("di Jalan X") hanya keterangan tempat.
      3. Pola maju ("lubang di Jalan X") tidak boleh melewati batas kalimat.
    """
    teks = re.sub(r"https?://\S+", " ", str(teks))
    if RE_IDIOM.search(teks):
        return False

    for m in RE_RUSAK.finditer(teks):
        if RE_MAJU.match(teks[m.end():m.end() + 40]):
            return True

        jendela = re.split(r"[.!?\n]", teks[max(0, m.start() - 90):m.start()])[-1]
        jalan = list(RE_JALAN.finditer(jendela))
        objek = list(RE_OBJEK.finditer(jendela))
        if not jalan:
            continue

        jalan_akhir = jalan[-1]
        objek_akhir = objek[-1] if objek else None
        jalan_lokatif = bool(RE_LOKATIF.search(jendela[:jalan_akhir.start()]))

        if objek_akhir is None:
            if not jalan_lokatif:
                return True
            continue

        objek_lokatif = bool(RE_LOKATIF.search(jendela[:objek_akhir.start()]))
        if objek_lokatif and not jalan_lokatif:
            return True
        if objek_akhir.start() > jalan_akhir.start() or jalan_lokatif:
            continue
        return True
    return False


print("Pola dan fungsi siap dipakai.")

Proses penggabungan & Data Filtering

In [ ]:
# Gabungkan semua berkas hasil scraping jadi satu tabel
df = pd.concat([pd.read_csv(f) for f in berkas], ignore_index=True)
log = []

def catat(nama):
    """Catat jumlah baris yang tersisa, dan berhenti bila datanya habis."""
    log.append((nama, len(df)))
    if len(df) == 0:
        raise SystemExit(f"Berhenti di tahap '{nama}': tidak ada baris yang tersisa.")

catat("1. Total data mentah")

# Hapus duplikat berdasarkan ID unggahan
df = df.drop_duplicates(subset="id_str")
catat("2. Setelah hapus duplikat ID")

# Buang retweet
df = df[~df["full_text"].astype(str).str.strip().str.startswith("RT @")]
catat("3. Setelah buang retweet")

# Buang duplikat
norm = df["full_text"].map(
    lambda t: re.sub(r"\s+", " ", re.sub(
        r"[^a-z0-9\s]", " ", re.sub(r"https?://\S+", "", str(t).lower()))).strip())
df = df[~norm.duplicated()]
catat("4. Setelah buang teks kembar")

# Filter berdasarkan tahun
df["tahun"] = pd.to_datetime(df["created_at"], format="mixed", utc=True).dt.year
df = df[df["tahun"] >= TAHUN_MIN]
catat(f"5. Tahun {TAHUN_MIN} ke atas")

# Gali lokasi dari isi teks, kemudian filter wilayah DKI Jakarta
teks = df["full_text"].astype(str)
df["wilayah"]    = teks.map(deteksi_wilayah)
df["sekitar"]    = teks.map(lambda t: cari_kota(t, KOTA_SEKITAR))
df["luar_jkt"]   = teks.map(lambda t: cari_kota(t, KOTA_JAUH))
df["nama_jalan"] = teks.map(ekstrak_nama_jalan)

df["status_lokasi"] = "tanpa lokasi"
df.loc[teks.str.lower().str.contains("jakarta|jkt|dki", na=False), "status_lokasi"] = "Jakarta (umum)"
df.loc[df["sekitar"]  != "", "status_lokasi"] = "sekitar Jakarta"
df.loc[df["luar_jkt"] != "", "status_lokasi"] = "luar Jabodetabek"
df.loc[df["wilayah"].str.startswith("AMBIGU"), "status_lokasi"] = "ambigu"
df.loc[df["wilayah"].str.startswith(("Jakarta", "Kep")), "status_lokasi"] = "DKI"

simpan = ["DKI", "Jakarta (umum)", "ambigu"] + (["sekitar Jakarta"] if WILAYAH_SEKITAR else [])
df = df[df["status_lokasi"].isin(simpan)]
catat("6. Wilayah DKI Jakarta")

# Memastikan yang rusak memang jalan, bukan benda lain di jalan
df["subjek_jalan"] = df["full_text"].astype(str).map(subjek_adalah_jalan)
df = df[df["subjek_jalan"]]
catat("7. Subjek kerusakan = jalan")

# Pisahkan keluhan warga dari publikasi perbaikan oleh instansi
t2 = df["full_text"].astype(str)
df["jenis"] = "keluhan"
df.loc[t2.str.contains(RE_PERBAIKAN) & ~t2.str.contains(RE_KELUHAN), "jenis"] = "laporan perbaikan"
df["trotoar"] = t2.str.contains(r"\btrotoar\b", case=False)
df["jpo"] = t2.str.contains(
    r"\b(?:jpo|jembatan penyeberangan|jembatan penyebrangan|flyover|fly over|jembatan layang)\b",
    case=False, regex=True)
df["tol"] = t2.str.contains(r"\b(?:tol|jorr|japek|wiyoto)\b", case=False, regex=True)

if BUANG_LAPORAN_PERBAIKAN:
    df = df[df["jenis"] == "keluhan"];  catat("8. Buang laporan perbaikan")
if BUANG_TROTOAR:
    df = df[~df["trotoar"]];            catat("9. Buang keluhan trotoar")
if BUANG_JPO:
    df = df[~df["jpo"]];                catat("10. Buang JPO dan flyover")
if BUANG_TOL:
    df = df[~df["tol"]];                catat("11. Buang jalan tol")

print("=== ALUR PENYARINGAN DATA ===")
for nama, jumlah in log:
    print(f"{nama:<34}: {jumlah:>6}")
print(f"\n>>> DATA AKHIR: {len(df)} baris")

Data Result

In [ ]:
KOLOM = ["id_str", "created_at", "tahun", "username", "lang",
         "full_text", "wilayah", "nama_jalan", "status_lokasi", "jenis", "tweet_url"]
KOLOM = [k for k in KOLOM if k in df.columns]
data_final = df[KOLOM].reset_index(drop=True)
data_final.to_csv("data_final.csv", index=False)

print(f"Jumlah baris : {len(data_final)}")
print(f"Jumlah kolom : {len(data_final.columns)}")
print("\n--- Kelengkapan tiap kolom ---")
for k in KOLOM:
    terisi = (data_final[k].astype(str).str.strip() != "").sum()
    print(f"{k:<16}: {terisi:>4} dari {len(data_final)}")

print("\n--- Sebaran wilayah ---")
print(data_final["wilayah"].replace("", "(tanpa wilayah)").value_counts().to_string())

print("\n--- Lima baris pertama ---")
data_final.head()

**Labelling**

In [ ]:
print("Pilih file: label_acuan_final.csv")
up = files.upload()
acuan = pd.read_csv(next(iter(up)))

hilang = set(acuan["id_str"]) - set(data_final["id_str"])
print(f"\nLabel acuan terbaca       : {len(acuan)} baris")
print(f"Cocok dengan data_final   : {len(acuan) - len(hilang)} baris")
if hilang:
    raise SystemExit(
        f"{len(hilang)} baris label tidak ditemukan di data_final.\n"
        "Biasanya karena file label pernah dibuka lalu disimpan ulang lewat Excel, "
        "sehingga angka id_str berubah. Pakai file label_acuan_final.csv yang asli.")

print("\n=== SEBARAN LABEL ACUAN ===")
for k in ["Tinggi", "Sedang", "Rendah"]:
    n = int((acuan["urgensi"] == k).sum())
    print(f"{k:<8}: {n:>4} baris  ({n / len(acuan) * 100:5.1f}%)")
acuan[["full_text", "urgensi"]].head()

In [ ]:
SINYAL_TINGGI = {
    "korban jiwa":   r"(?:tewas|newas|meninggal|wafat|korban jiwa|meregang nyawa)",
    "korban luka":   r"(?:luka|dilarikan ke rumah sakit|patah tulang|dirawat)",
    "kecelakaan":    r"(?:kecelak\W*an|laka|terjatuh|jatuh|terpelanting|tertabrak|menabrak|terperosok|kejeblos|kejebos|oleng)",
    "jalan ambles":  r"(?:ambles|amblas|amblus|longsor|runtuh|ambruk|badan jalan hilang)",
    "lubang parah":  r"(?:lubang\s+(?:dalam|besar|menganga|raksasa)|menganga|sedalam|kubangan)",
    "jalan ditutup": r"(?:ditutup|penutupan jalan|pengalihan rute|penyesuaian rute|dialihkan|rekayasa lalu lintas)",
}
SINYAL_SEDANG = {
    "rawan atau bahaya":    r"(?:rawan|bahaya|membahayakan|berbahaya|mengancam|ancaman|celaka)",
    "imbauan hati-hati":    r"(?:hati\W*hati|hati\W*2|hati\W*²|waspada|awas|dihimbau|diimbau|pelan\W*pelan)",
    "gangguan lalu lintas": r"(?:macet|tersendat|padat|penyempitan|menyempit|antrean|antrian|kemacetan|menghindari)",
    "permintaan perbaikan": r"(?:tolong|mohon|segera|dibenerin|harap cepat|kapan diperbaiki|tak kunjung|belum juga|jangan tunggu)",
    "kerusakan parah":      r"(?:rusak parah|parah|hancur|ancur)",
}
SINYAL_RENDAH = {
    "kerusakan ringan":    r"(?:retak|bergelombang|lubang kecil|keriting|gronjal)",
    "pemberitaan agregat": r"(?:\d[\d\.\,]*\s*(?:titik|ruas|km|kilometer)|persen)",
}


def beri_label(teks):
    """Tentukan urgensi dari kata penanda. Tinggi menang atas Sedang, Sedang atas Rendah."""
    t = re.sub(r"https?://\S+", " ", str(teks))
    for nama, kamus in (("Tinggi", SINYAL_TINGGI), ("Sedang", SINYAL_SEDANG), ("Rendah", SINYAL_RENDAH)):
        if any(re.search(p, t, re.I) for p in kamus.values()):
            return nama
    return None


sisa = data_final[~data_final["id_str"].isin(set(acuan["id_str"]))].copy()
sisa["urgensi"] = sisa["full_text"].map(beri_label)
sisa_label = sisa[sisa["urgensi"].notna()].copy()

print("=== PELABELAN OTOMATIS PADA SISA DATA ===")
print(f"Baris di luar acuan      : {len(sisa)}")
print(f"Berhasil dilabeli aturan : {len(sisa_label)}")
print(f"Tanpa penanda, dilewati  : {len(sisa) - len(sisa_label)}")
print("\nSebaran:")
print(sisa_label["urgensi"].value_counts().to_string())

In [ ]:
a = data_final.merge(acuan[["id_str", "urgensi"]], on="id_str", how="inner")
a["sumber_label"] = "manusia"
sisa_label["sumber_label"] = "aturan"

dataset = pd.concat([a, sisa_label], ignore_index=True)
dataset.to_csv("dataset_berlabel.csv", index=False)

print(f"Total baris : {len(dataset)}")
print(dataset["sumber_label"].value_counts().to_string())
print("\nSebaran urgensi:")
for k in ["Tinggi", "Sedang", "Rendah"]:
    n = int((dataset["urgensi"] == k).sum())
    print(f"  {k:<8}: {n:>4}  ({n / len(dataset) * 100:5.1f}%)")

files.download("dataset_berlabel.csv")

In [ ]:
tidak_terpakai = data_final[~data_final["id_str"].isin(set(dataset["id_str"]))]
print("Tidak masuk dataset:", len(tidak_terpakai), "baris")
tidak_terpakai.to_csv("tidak_terpakai.csv", index=False)
tidak_terpakai[["full_text"]].head(10)